# True Anti-Fabrication Harness Wrapper (V4)

This notebook acts as the cryptographically secure runner for `verify_kaggle.py`. It uses the true `harness_core.py` from the anti-fabrication toolkit to plant canaries, run the subprocess, verify the output, and generate the HMAC verification card.

In [ ]:
!pip install -q ultralytics timm opencv-python-headless pycocotools cryptography

In [ ]:
import os
import sys
import shutil
import glob
import secrets
import re
import json

# 1. Find the Anti-Fabrication Toolkit
toolkit_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'harness_core.py' in files:
        toolkit_path = root
        break

assert toolkit_path, "FATAL: anti_fabrication_toolkit not found! Did you add it as a dataset?"
sys.path.insert(0, toolkit_path)
print(f"Found Toolkit at: {toolkit_path}")

# 2. Find the eval script (verify_kaggle.py)
script_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'verify_kaggle.py' in files:
        script_path = os.path.join(root, 'verify_kaggle.py')
        break

assert script_path, "FATAL: verify_kaggle.py not found!"

# Copy to working directory because Kaggle input is read-only
shutil.copy(script_path, '/kaggle/working/verify_kaggle.py')
local_script = '/kaggle/working/verify_kaggle.py'

# 3. Setup Dataset (Must copy images to working dir so we can plant canaries)
DATASET_ROOT = '/kaggle/input/polypdb-polyp-raw/PolypDB'
assert os.path.exists(DATASET_ROOT), f"FATAL: DATASET_ROOT {DATASET_ROOT} not found!"

WORKING_DATASET = '/kaggle/working/dataset_copy'
if os.path.exists(WORKING_DATASET):
    shutil.rmtree(WORKING_DATASET)
shutil.copytree(DATASET_ROOT, WORKING_DATASET)
print(f"Dataset copied to: {WORKING_DATASET}")

In [ ]:
from harness_core import plant_multi_canary, run_live, verify, sign_verdict

# 4. Plant Canaries
print("[*] Planting Canaries...")
secret = plant_multi_canary(WORKING_DATASET)

# 5. Execute Subprocess (run_live handles the random nonce automatically in older versions, but we pass it just in case)
print(f"[*] Generated Random Nonce: {secret['nonce']}")

# We must pass the correct environment variables so verify_kaggle.py knows what to evaluate
env_vars = os.environ.copy()
env_vars["VERIFY_NONCE"] = secret['nonce']
env_vars["VERIFY_DATASET_NAME"] = "PolypDB"
env_vars["VERIFY_DATASET_ROOT"] = WORKING_DATASET

print("[*] Launching Subprocess...")
stdout_text, arrivals, elapsed_sec, retcode = run_live(local_script, extra_env=env_vars, timeout_sec=10000)

print("[*] Subprocess Finished. Raw Output Snippet:")
print("\n".join(stdout_text.splitlines()[-20:]))


In [ ]:
# 6. Cryptographic Verification
print("\n[*] Verifying Subprocess Output...")

img_paths = glob.glob(os.path.join(WORKING_DATASET, "images", "*.*", ))
img_paths = [p for p in img_paths if p.endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp'))]
real_total_images_on_disk = len(img_paths)

match = re.search(r"Total Evaluated Images:\s*(\d+)", stdout_text)
claimed_total = int(match.group(1)) if match else 0

try:
    result = verify(
        full_text=stdout_text,
        arrivals=arrivals,
        elapsed_sec=elapsed_sec,
        secret=secret,
        claimed_total_images=claimed_total,
        real_total_images_on_disk=real_total_images_on_disk
    )
    print("\n============================================================")
    if result['pass']:
        print("VERIFICATION SUCCESSFUL!")
    else:
        print("VERIFICATION FAILED! SEE FINDINGS.")
    print("============================================================")
    
    for finding in result['findings']:
        print(finding)
        
    # Generate the Completion Card
    card_path = '/kaggle/working/completion_card.json'
    signed = sign_verdict(result, secret['hmac_key'])
    card = {
        "decision": "ADMITTED" if result['pass'] else "WITHHELD",
        "verdict": signed["verdict"],
        "signature": signed["signature"]
    }
    with open(card_path, 'w') as f:
        json.dump(card, f, indent=2)
    print(f"\n[*] Card written to {card_path}")
    print(json.dumps(card, indent=2))
    
except Exception as e:
    print("\n[FATAL ERROR] VERIFICATION FAILED!")
    print(str(e))
    sys.exit(1)
